In [ ]:
!pip install awscli boto3

In [ ]:
import glob
import os
import zipfile
import tarfile
import boto3
from google.colab import drive, userdata

In [ ]:
drive.mount('/content/drive')

In [ ]:
# Load secrets from collab user data
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = userdata.get('AWS_DEFAULT_REGION')

In [ ]:
# Configuration
DRIVE_ZIP_FOLDER = "/content/drive/My Drive/Academia/Birkbeck/Project/Data/Activity Net"
BUCKET_NAME = "video-sum-pipeline"
S3_PREFIX = "raw-data/SM-MrHiSum and SM-VideoXum/ActivityNet-Data/"

def get_existing_s3_keys(s3_client):
    """Fetch list of existing files in S3 bucket prefix."""
    print("Fetching list of already uploaded files from S3 to resume safely...")
    paginator = s3_client.get_paginator('list_objects_v2')
    existing_keys = set()

    # Paginate through the bucket, adding keys to our set
    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix=S3_PREFIX):
        if 'Contents' in page:
            for obj in page['Contents']:
                existing_keys.add(obj['Key'])

    print(f"Found {len(existing_keys)} files already in S3.")
    return existing_keys

def stream_archives_to_s3():
    s3_client = boto3.client('s3')

    # Build index of existing files
    existing_keys = get_existing_s3_keys(s3_client)

    archive_files = []
    for ext in ("*.zip", "*.tar.gz", "*.tar"):
        archive_files.extend(glob.glob(os.path.join(DRIVE_ZIP_FOLDER, ext)))

    # Sort list alphabetically to ensure consistent processing order
    archive_files = sorted(archive_files)

    if not archive_files:
        print(f"No archives found in {DRIVE_ZIP_FOLDER}")
        return

    print(
        f"Found {len(archive_files)} archive files. Initialising resumption pipeline..."
        )

    # Iterate files to stream to S3
    for index, archive_path in enumerate(archive_files):
        archive_filename = os.path.basename(archive_path)
        print(f"Item {index} Streaming: {archive_filename} directly to S3")

        uploaded_count = 0
        skipped_count = 0

        try:
            if archive_filename.endswith(".zip"):
                with zipfile.ZipFile(archive_path, 'r') as zf:
                    for member in zf.infolist():
                        if not member.is_dir():
                            s3_key = f"{S3_PREFIX}{os.path.basename(member.filename)}"

                            # Check if the file is already in S3
                            if s3_key in existing_keys:
                                skipped_count += 1
                                continue

                            with zf.open(member) as file_obj:
                                s3_client.upload_fileobj(file_obj, BUCKET_NAME, s3_key)

                            uploaded_count += 1
                            if uploaded_count % 100 == 0:
                                print(f"[{archive_filename}] Uploaded {uploaded_count} new files (Skipped {skipped_count})...")

            elif archive_filename.endswith((".tar.gz", ".tgz", ".tar")):
                mode = 'r:gz' if archive_filename.endswith("gz") else 'r'
                with tarfile.open(archive_path, mode) as tf:

                    member = tf.next()
                    while member is not None:
                        if member.isfile():
                            s3_key = f"{S3_PREFIX}{os.path.basename(member.name)}"

                            # Check if the file is already in S3
                            if s3_key in existing_keys:
                                skipped_count += 1
                            else:
                                file_obj = tf.extractfile(member)
                                if file_obj:
                                    s3_client.upload_fileobj(file_obj, BUCKET_NAME, s3_key)
                                    uploaded_count += 1

                                    if uploaded_count % 100 == 0:
                                        print(f"[{archive_filename}] Uploaded {uploaded_count} new files (Skipped {skipped_count})...")

                        member = tf.next()

            print(f"{archive_filename} Complete! Uploaded: {uploaded_count} | Skipped: {skipped_count}")

        except Exception as e:
            print(f"{archive_filename} ERROR during processing: {e}")

    print("All archives streamed to S3 successfully!")

# Execute the pipeline
stream_archives_to_s3()